# 33_gold_marts — Gold Business Marts + Wrapped

Builds all business-facing mart tables from the Gold star schema.
All marts use **full recompute with overwrite** (tens of thousands to low millions of rows,
idempotent by construction; enterprise scale would use CDF-incremental or Materialized Views).

| Table | Business Question |
|---|---|
| `peak_usage_patterns` | Q1: When do users listen? (capacity planning) |
| `subscription_propensity_segments` | Q2: Which free users behave like premium? |
| `content_performance_trends` | Q3: Which genres/artists are rising or falling? |
| `wrapped_user_year` | Consumer Wrapped: one flat row per user per year |
| `audit_pipeline_runs` | Pipeline health by day/layer/table |
| `audit_quality_summary` | Quarantine rates by rule (counts only, no raw_record) |
| `audit_freshness` | Latest load timestamp and row count per layer/table |

In [ ]:
%run ./00_config


## Mart 1: Peak Usage Patterns (Q1 — Capacity Planning)

In [ ]:
from datetime import datetime, timezone

_t0 = datetime.now(timezone.utc)
try:
    # Join fact to dim_user to get timezone for local_hour calculation
    df_peak = spark.sql(f"""
        SELECT
            f.hour_of_day,
            d.day_of_week,
            dev.device_category,
            f.country,
            COUNT(*)                                                     AS utc_plays,
            SUM(f.minutes_played)                                        AS utc_minutes,
            -- local_hour: convert UTC played_at using user's timezone
            HOUR(from_utc_timestamp(
                DATE_ADD(TO_DATE(CAST(f.date_key AS STRING),'yyyyMMdd'),0),
                COALESCE(u.timezone, 'UTC')))                            AS local_hour,
            CURRENT_TIMESTAMP()                                          AS load_timestamp
        FROM {table_ref(GOLD['fact_listening'])} f
        JOIN {table_ref(GOLD['dim_date'])} d ON f.date_key = d.date_key
        JOIN {table_ref(GOLD['dim_device'])} dev ON f.device_sk = dev.device_sk
        LEFT JOIN {table_ref(GOLD['dim_user'])} u ON f.user_sk = u.user_sk
        WHERE f.minutes_played IS NOT NULL
        GROUP BY f.hour_of_day, d.day_of_week, dev.device_category,
                 f.country, u.timezone
    """)
    n = df_peak.count()
    df_peak.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['peak_usage_patterns'])
    log_run('SILVER_TO_GOLD','gold.peak_usage_patterns','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'peak_usage_patterns: {n:,} rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.peak_usage_patterns','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## Mart 2: Subscription Propensity Segments (Q2)

In [ ]:
_t0 = datetime.now(timezone.utc)
try:
    # Compute engagement score components per user
    # discovery_rate: share of plays on a track NOT played by that user in prev 60 days
    # For discovery, we use Silver events directly (cheaper) and flag via LAG window
    df_prop = spark.sql(f"""
        WITH user_stats AS (
            SELECT
                f.user_sk,
                u.user_id,
                u.is_premium,
                SUM(f.minutes_played)                          AS total_minutes,
                COUNT(DISTINCT f.session_id)                   AS session_count,
                COUNT(DISTINCT t.genre)                        AS genre_diversity,
                AVG(f.is_skipped)                              AS skip_rate,
                COUNT(DISTINCT f.device_sk)                    AS raw_device_count
            FROM {table_ref(GOLD['fact_listening'])} f
            JOIN {table_ref(GOLD['dim_user'])} u ON f.user_sk=u.user_sk AND u.is_current=TRUE
            JOIN {table_ref(GOLD['dim_track'])} t ON f.track_sk=t.track_sk
            GROUP BY f.user_sk, u.user_id, u.is_premium
        ),
        discovery AS (
            SELECT user_id,
                   AVG(CASE WHEN is_discovery THEN 1.0 ELSE 0.0 END) AS discovery_rate
            FROM (
                SELECT e.user_id, e.track_id, e.played_at,
                       NOT EXISTS (
                           SELECT 1 FROM {table_ref(SILVER_EVENTS)} prev
                           WHERE prev.user_id = e.user_id
                             AND prev.track_id = e.track_id
                             AND prev.played_at < e.played_at
                             AND prev.played_at >= DATE_ADD(e.played_at, -{DISCOVERY_LOOKBACK_DAYS})
                       ) AS is_discovery
                FROM {table_ref(SILVER_EVENTS)} e
                WHERE e.data_source <> 'real_api'
            ) sub
            GROUP BY user_id
        ),
        vol_norm AS (
            SELECT user_id,
                   (LOG(total_minutes + 1) - mn) / NULLIF(mx - mn, 0) AS volume_norm
            FROM (
                SELECT u2.user_id, SUM(f2.minutes_played) AS total_minutes,
                       MIN(LOG(SUM(f2.minutes_played)+1)) OVER() AS mn,
                       MAX(LOG(SUM(f2.minutes_played)+1)) OVER() AS mx
                FROM {table_ref(GOLD['fact_listening'])} f2
                JOIN {table_ref(GOLD['dim_user'])} u2 ON f2.user_sk=u2.user_sk AND u2.is_current=TRUE
                GROUP BY u2.user_id
            ) v
        )
        SELECT
            s.user_id,
            s.is_premium,
            s.total_minutes,
            s.session_count,
            s.genre_diversity,
            COALESCE(s.skip_rate, 0)                             AS skip_rate,
            -- device_diversity: normalise entropy (raw_device_count / max)
            s.raw_device_count * 1.0 /
                NULLIF(MAX(s.raw_device_count) OVER(), 0)        AS device_diversity,
            COALESCE(d.discovery_rate, 0)                        AS discovery_rate,
            COALESCE(vn.volume_norm, 0)                          AS volume_norm,
            -- engagement_score = W1*volume + W2*device + W3*(1-skip) + W4*discovery
            {W1_VOLUME}  * COALESCE(vn.volume_norm, 0) +
            {W2_DEVICE_DIVERSITY} * (s.raw_device_count * 1.0 /
                NULLIF(MAX(s.raw_device_count) OVER(), 0)) +
            {W3_COMPLETION} * (1 - COALESCE(s.skip_rate, 0)) +
            {W4_DISCOVERY} * COALESCE(d.discovery_rate, 0)      AS engagement_score,
            CASE
                WHEN ({W1_VOLUME}  * COALESCE(vn.volume_norm,0) +
                      {W2_DEVICE_DIVERSITY} * (s.raw_device_count * 1.0 /
                          NULLIF(MAX(s.raw_device_count) OVER(),0)) +
                      {W3_COMPLETION} * (1-COALESCE(s.skip_rate,0)) +
                      {W4_DISCOVERY} * COALESCE(d.discovery_rate,0))
                     >= {PROPENSITY_TIERS[0]} THEN 'High'
                WHEN ({W1_VOLUME}  * COALESCE(vn.volume_norm,0) +
                      {W2_DEVICE_DIVERSITY} * (s.raw_device_count * 1.0 /
                          NULLIF(MAX(s.raw_device_count) OVER(),0)) +
                      {W3_COMPLETION} * (1-COALESCE(s.skip_rate,0)) +
                      {W4_DISCOVERY} * COALESCE(d.discovery_rate,0))
                     >= {PROPENSITY_TIERS[1]} THEN 'Medium'
                ELSE 'Low'
            END AS tier,
            (s.is_premium = FALSE OR s.is_premium IS NULL)
                AND ({W1_VOLUME}  * COALESCE(vn.volume_norm,0) +
                     {W2_DEVICE_DIVERSITY} * (s.raw_device_count * 1.0 /
                         NULLIF(MAX(s.raw_device_count) OVER(),0)) +
                     {W3_COMPLETION} * (1-COALESCE(s.skip_rate,0)) +
                     {W4_DISCOVERY} * COALESCE(d.discovery_rate,0))
                     >= {HIGH_PROPENSITY_THRESHOLD}             AS is_high_conversion_propensity,
            CURRENT_TIMESTAMP()                                  AS load_timestamp
        FROM user_stats s
        LEFT JOIN discovery d ON s.user_id = d.user_id
        LEFT JOIN vol_norm  vn ON s.user_id = vn.user_id
    """)
    n = df_prop.count()
    df_prop.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['subscription_propensity_segments'])
    flagged = df_prop.filter(F.col('is_high_conversion_propensity')==True).count()
    log_run('SILVER_TO_GOLD','gold.subscription_propensity_segments','overwrite',
            _t0,'SUCCESS',rows_read=n,rows_inserted=n)
    print(f'subscription_propensity_segments: {n:,} users | {flagged} High Conversion free users')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.subscription_propensity_segments','overwrite',
            _t0,'FAILURE',error_message=str(_e))
    raise


## Mart 3: Content Performance Trends (Q3 — Rising/Falling)

In [ ]:
_t0 = datetime.now(timezone.utc)
try:
    df_trends = spark.sql(f"""
        WITH monthly AS (
            SELECT
                CAST(DATE_FORMAT(DATE(CONCAT(CAST(d.year AS STRING),'-',
                    LPAD(CAST(d.month AS STRING),2,'0'),'-01')),'yyyyMM') AS INT) AS year_month,
                'genre'                        AS group_type,
                t.genre                        AS group_name,
                SUM(f.minutes_played)          AS monthly_minutes
            FROM {table_ref(GOLD['fact_listening'])} f
            JOIN {table_ref(GOLD['dim_date'])} d ON f.date_key=d.date_key
            JOIN {table_ref(GOLD['dim_track'])} t ON f.track_sk=t.track_sk
            WHERE f.minutes_played IS NOT NULL
            GROUP BY 1, 2, 3
            UNION ALL
            SELECT
                CAST(DATE_FORMAT(DATE(CONCAT(CAST(d.year AS STRING),'-',
                    LPAD(CAST(d.month AS STRING),2,'0'),'-01')),'yyyyMM') AS INT) AS year_month,
                'artist'                       AS group_type,
                t.artist_name                  AS group_name,
                SUM(f.minutes_played)          AS monthly_minutes
            FROM {table_ref(GOLD['fact_listening'])} f
            JOIN {table_ref(GOLD['dim_date'])} d ON f.date_key=d.date_key
            JOIN {table_ref(GOLD['dim_track'])} t ON f.track_sk=t.track_sk
            WHERE f.minutes_played IS NOT NULL
            GROUP BY 1, 2, 3
        ),
        totals AS (
            SELECT year_month, group_type,
                   SUM(monthly_minutes) AS total_minutes
            FROM monthly GROUP BY year_month, group_type
        ),
        with_share AS (
            SELECT m.year_month, m.group_type, m.group_name, m.monthly_minutes,
                   m.monthly_minutes / NULLIF(t.total_minutes, 0) AS monthly_share
            FROM monthly m JOIN totals t
              ON m.year_month=t.year_month AND m.group_type=t.group_type
        )
        SELECT
            year_month, group_type, group_name, monthly_minutes,
            monthly_share,
            monthly_share - LAG(monthly_share) OVER (
                PARTITION BY group_type, group_name ORDER BY year_month
            ) AS mom_delta,
            CURRENT_TIMESTAMP() AS load_timestamp
        FROM with_share
    """)
    n = df_trends.count()
    df_trends.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['content_performance_trends'])
    log_run('SILVER_TO_GOLD','gold.content_performance_trends','overwrite',
            _t0,'SUCCESS',rows_read=n,rows_inserted=n)
    print(f'content_performance_trends: {n:,} rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.content_performance_trends','overwrite',
            _t0,'FAILURE',error_message=str(_e))
    raise


## Mart 4: Wrapped User Year

One flat row per user per year. No arrays — flat columns for Power BI compatibility.
Mirrors the `Mixtape_Wrapped_storyboard.html` data shape.

In [ ]:
_t0 = datetime.now(timezone.utc)
try:
    df_wrapped = spark.sql(f"""
        WITH base AS (
            SELECT
                u.user_id,
                d.year,
                t.artist_name,
                CONCAT(t.track_name, ' - ', t.artist_name) AS track_label,
                t.genre,
                f.minutes_played,
                f.play_count,
                f.is_skipped,
                f.hour_of_day,
                f.track_sk,
                f.session_id,
                f.data_source
            FROM {table_ref(GOLD['fact_listening'])} f
            JOIN {table_ref(GOLD['dim_user'])}  u ON f.user_sk=u.user_sk AND u.is_current=TRUE
            JOIN {table_ref(GOLD['dim_date'])}  d ON f.date_key=d.date_key
            JOIN {table_ref(GOLD['dim_track'])} t ON f.track_sk=t.track_sk
            WHERE f.minutes_played IS NOT NULL
        ),
        artist_ranks AS (
            SELECT user_id, year, artist_name,
                   SUM(minutes_played) AS mins,
                   ROW_NUMBER() OVER (PARTITION BY user_id, year ORDER BY SUM(minutes_played) DESC) AS rk
            FROM base GROUP BY user_id, year, artist_name
        ),
        track_ranks AS (
            SELECT user_id, year, track_label,
                   SUM(minutes_played) AS mins,
                   ROW_NUMBER() OVER (PARTITION BY user_id, year ORDER BY SUM(minutes_played) DESC) AS rk
            FROM base GROUP BY user_id, year, track_label
        ),
        genre_ranks AS (
            SELECT user_id, year, genre,
                   COUNT(*) AS plays,
                   ROW_NUMBER() OVER (PARTITION BY user_id, year ORDER BY COUNT(*) DESC) AS rk
            FROM base WHERE genre IS NOT NULL AND genre <> 'unclassified'
            GROUP BY user_id, year, genre
        ),
        peak_hours AS (
            SELECT user_id, year, hour_of_day,
                   ROW_NUMBER() OVER (PARTITION BY user_id, year ORDER BY COUNT(*) DESC) AS rk
            FROM base GROUP BY user_id, year, hour_of_day
        ),
        totals AS (
            SELECT user_id, year,
                   SUM(minutes_played) AS total_minutes,
                   SUM(play_count)     AS total_plays
            FROM base GROUP BY user_id, year
        ),
        discovery AS (
            SELECT user_id, year,
                   AVG(CASE WHEN data_source <> 'real_api' THEN
                       CASE WHEN is_skipped IS NULL THEN NULL ELSE 1.0 - is_skipped END
                   END) AS discovery_rate  -- proxy from fact; real discovery in propensity mart
            FROM base GROUP BY user_id, year
        )
        SELECT
            t1.user_id,
            t1.year,
            tot.total_minutes,
            tot.total_plays,
            MAX(CASE WHEN ar.rk=1 THEN ar.artist_name END) AS top_artist_1,
            MAX(CASE WHEN ar.rk=2 THEN ar.artist_name END) AS top_artist_2,
            MAX(CASE WHEN ar.rk=3 THEN ar.artist_name END) AS top_artist_3,
            MAX(CASE WHEN ar.rk=4 THEN ar.artist_name END) AS top_artist_4,
            MAX(CASE WHEN ar.rk=5 THEN ar.artist_name END) AS top_artist_5,
            MAX(CASE WHEN gr.rk=1 THEN gr.genre END) AS top_genre_1,
            MAX(CASE WHEN gr.rk=2 THEN gr.genre END) AS top_genre_2,
            MAX(CASE WHEN gr.rk=3 THEN gr.genre END) AS top_genre_3,
            MAX(CASE WHEN tr.rk=1 THEN tr.track_label END) AS top_track_1,
            MAX(CASE WHEN tr.rk=2 THEN tr.track_label END) AS top_track_2,
            MAX(CASE WHEN tr.rk=3 THEN tr.track_label END) AS top_track_3,
            MAX(CASE WHEN tr.rk=4 THEN tr.track_label END) AS top_track_4,
            MAX(CASE WHEN tr.rk=5 THEN tr.track_label END) AS top_track_5,
            d.discovery_rate,
            MAX(CASE WHEN ph.rk=1 THEN ph.hour_of_day END) AS peak_hour,
            CURRENT_TIMESTAMP() AS load_timestamp
        FROM totals tot
        JOIN (SELECT DISTINCT user_id, year FROM base) t1 ON tot.user_id=t1.user_id AND tot.year=t1.year
        LEFT JOIN artist_ranks ar ON t1.user_id=ar.user_id AND t1.year=ar.year AND ar.rk <= 5
        LEFT JOIN track_ranks  tr ON t1.user_id=tr.user_id AND t1.year=tr.year AND tr.rk <= 5
        LEFT JOIN genre_ranks  gr ON t1.user_id=gr.user_id AND t1.year=gr.year AND gr.rk <= 3
        LEFT JOIN peak_hours   ph ON t1.user_id=ph.user_id AND t1.year=ph.year AND ph.rk = 1
        LEFT JOIN discovery     d ON t1.user_id=d.user_id  AND t1.year=d.year
        GROUP BY t1.user_id, t1.year, tot.total_minutes, tot.total_plays, d.discovery_rate
    """)
    n = df_wrapped.count()
    df_wrapped.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['wrapped_user_year'])
    log_run('SILVER_TO_GOLD','gold.wrapped_user_year','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'wrapped_user_year: {n:,} user-year rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.wrapped_user_year','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## Audit Mart 1: Pipeline Runs

In [ ]:
_t0 = datetime.now(timezone.utc)
try:
    df_runs = spark.sql(f"""
        SELECT
            DATE(start_time)      AS run_date,
            layer,
            target_table,
            COUNT(*)              AS run_count,
            SUM(CASE WHEN status='SUCCESS' THEN 1 ELSE 0 END) AS success_count,
            SUM(CASE WHEN status='FAILURE' THEN 1 ELSE 0 END) AS failure_count,
            SUM(COALESCE(rows_read,0))     AS rows_read_sum,
            SUM(COALESCE(rows_inserted,0)
                + COALESCE(rows_updated,0))  AS rows_out_sum,
            PERCENTILE_APPROX(duration_sec,0.5) AS duration_p50,
            MAX(duration_sec)               AS duration_max,
            CURRENT_TIMESTAMP()             AS load_timestamp
        FROM {table_ref(EXECUTION_LOG)}
        GROUP BY DATE(start_time), layer, target_table
    """)
    n = df_runs.count()
    df_runs.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['audit_pipeline_runs'])
    log_run('SILVER_TO_GOLD','gold.audit_pipeline_runs','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'audit_pipeline_runs: {n:,} rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.audit_pipeline_runs','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## Audit Mart 2: Quality Summary (counts only — no raw_record)

In [ ]:
_t0 = datetime.now(timezone.utc)
try:
    # Bronze quarantine counts by _reason
    df_bq = spark.sql(f"""
        SELECT _partition_dt AS partition_dt, 'bronze' AS layer, _reason AS dq_rule,
               COUNT(*) AS rejection_count
        FROM {table_ref(BRONZE_EVENTS_QUARANTINE)}
        GROUP BY _partition_dt, _reason
    """)
    # Silver quarantine counts by _dq_rule
    df_sq = spark.sql(f"""
        SELECT CAST(NULL AS DATE) AS partition_dt, 'silver' AS layer, _dq_rule AS dq_rule,
               COUNT(*) AS rejection_count
        FROM {table_ref(SILVER_EVENTS_QUARANTINE)}
        GROUP BY _dq_rule
    """)
    # Total events per partition for bronze rate
    df_bq_tot = spark.sql(f"""
        SELECT _partition_dt, COUNT(*) AS total
        FROM {table_ref(BRONZE_EVENTS)} GROUP BY _partition_dt
    """)

    df_qual = df_bq.join(df_bq_tot, df_bq['partition_dt']==df_bq_tot['_partition_dt'],'left') \
        .withColumn('total_count', F.coalesce(F.col('total'), F.lit(0))) \
        .withColumn('quarantine_rate', F.col('rejection_count') / F.greatest(F.col('total_count'),F.lit(1))) \
        .withColumn('above_alert', F.col('quarantine_rate') > F.lit(QUARANTINE_ALERT_RATE)) \
        .withColumn('load_timestamp', F.current_timestamp()) \
        .select('partition_dt','layer','dq_rule','rejection_count','total_count',
                'quarantine_rate','above_alert','load_timestamp') \
        .union(
            df_sq.withColumn('total_count', F.lit(0).cast('bigint'))
                 .withColumn('quarantine_rate', F.lit(None).cast('double'))
                 .withColumn('above_alert', F.lit(None).cast('boolean'))
                 .withColumn('load_timestamp', F.current_timestamp())
                 .select('partition_dt','layer','dq_rule','rejection_count','total_count',
                         'quarantine_rate','above_alert','load_timestamp')
        )

    n = df_qual.count()
    df_qual.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['audit_quality_summary'])
    log_run('SILVER_TO_GOLD','gold.audit_quality_summary','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'audit_quality_summary: {n:,} rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.audit_quality_summary','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## Audit Mart 3: Data Freshness

In [ ]:
_t0 = datetime.now(timezone.utc)
try:
    tables = [
        ('bronze', 'bronze.events',           BRONZE_EVENTS,       '_partition_dt'),
        ('silver', 'silver.events',            SILVER_EVENTS,       '_partition_dt'),
        ('silver', 'silver.users',             SILVER_USERS,        None),
        ('silver', 'silver.tracks',            SILVER_TRACKS,       None),
        ('gold',   'gold.fact_listening',      GOLD['fact_listening'], None),
        ('gold',   'gold.dim_user',            GOLD['dim_user'],      None),
    ]
    rows = []
    for layer, tname, tref, partcol in tables:
        df_t = spark.table(tref)
        cnt = df_t.count()
        max_ts = df_t.agg(F.max('load_timestamp')).first()[0]
        max_part = df_t.agg(F.max(partcol)).first()[0] if partcol else None
        rows.append((layer, tname, max_part, max_ts, cnt))
    from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType, LongType
    schema = StructType([
        StructField('layer', StringType()), StructField('table_name', StringType()),
        StructField('latest_partition', DateType()), StructField('latest_load_ts', TimestampType()),
        StructField('row_count', LongType())
    ])
    df_fresh = spark.createDataFrame(rows, schema).withColumn('load_timestamp', F.current_timestamp())
    n = df_fresh.count()
    df_fresh.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['audit_freshness'])
    log_run('SILVER_TO_GOLD','gold.audit_freshness','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'audit_freshness: {n:,} rows')
    display(df_fresh)
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.audit_freshness','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## Proof: Mart Row Counts + Propensity Check

In [ ]:
for m in ['peak_usage_patterns','subscription_propensity_segments',
           'content_performance_trends','wrapped_user_year',
           'audit_pipeline_runs','audit_quality_summary','audit_freshness']:
    print(f'{GOLD[m]}: {spark.table(GOLD[m]).count():,} rows')

print('\n--- Propensity tier distribution ---')
display(spark.sql(f"""
    SELECT tier, is_premium, COUNT(*) AS users,
           ROUND(AVG(engagement_score),3) AS avg_score
    FROM {table_ref(GOLD['subscription_propensity_segments'])}
    GROUP BY tier, is_premium ORDER BY tier, is_premium DESC
"""))

print('\n--- Wrapped sample (top user) ---')
display(spark.sql(f"""
    SELECT user_id, year, ROUND(total_minutes,0) AS mins, total_plays,
           top_artist_1, top_genre_1, top_track_1, peak_hour
    FROM {table_ref(GOLD['wrapped_user_year'])}
    ORDER BY total_minutes DESC LIMIT 5
"""))
